# 1. Environment & Workspace Setup

## 1.1 Install All Required Libraries

In [3]:
# Install core RAG framework and sub-packages
!pip install -q -U langchain langchain-community langchain-openai langchain-huggingface \
    langchain-text-splitters ragas chromadb pypdf sentence-transformers \
    fastembed rank_bm25 huggingface_hub

    # Force-install all required modular packages for 2026 RAG
!pip install -q -U langchain langchain-community langchain-text-splitters \
    langchain-huggingface langchain-openai pypdf chromadb \
    fastembed rank_bm25 jq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1.2 Secure Your Workspace

In [4]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define your project paths
# Change "CyberRAG_Project" to whatever you like
BASE_DIR = "/content/drive/MyDrive/CyberRAG_Project"
DATA_DIR = os.path.join(BASE_DIR, "data")
DB_DIR = os.path.join(BASE_DIR, "vector_store")

# 3. Create folders if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

print(f"Workspace ready at: {BASE_DIR}")
print(f"Action Required: Upload your PDF guidelines to: {DATA_DIR}")

Mounted at /content/drive
Workspace ready at: /content/drive/MyDrive/CyberRAG_Project
Action Required: Upload your PDF guidelines to: /content/drive/MyDrive/CyberRAG_Project/data


# 2. Selecting and Loading Data

## 2.1 & 2.2: Unified Ingestion (PDFs + JSON)

In [5]:
import os
import requests
import json
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Download and save your STIX JSON directly to your data folder
def download_stix_json(target_path):
    url = "https://www.cisa.gov/sites/default/files/2025-06/AA23-352A_StopRansomware-Play-Ransomware.stix_JSON.json"
    response = requests.get(url)
    if response.status_code == 200:
        with open(target_path, 'w', encoding='utf-8') as f:
            json.dump(response.json(), f, indent=4)
        print(f"STIX JSON saved to: {target_path}")
    else:
        print(f"Failed to download STIX JSON. Code: {response.status_code}")

json_file_path = os.path.join(DATA_DIR, 'threat_intel_play_ransomware_2025.json')
download_stix_json(json_file_path)

# 2. Ingestion Function for both PDFs and JSON
def load_all_cyber_knowledge(pdf_dir):
    print(f"Scanning Knowledge Base")

    # Load PDFs (NIST, ISO, CISA, OWASP)
    pdf_loader = DirectoryLoader(pdf_dir, glob="./*.pdf", loader_cls=PyPDFLoader)
    pdf_docs = pdf_loader.load()

    # Load the STIX JSON (Extracting technical 'description' fields)
    json_loader = JSONLoader(
        file_path=json_file_path,
        jq_schema='.objects[] | select(.description != null) | .description',
        text_content=True
    )
    json_docs = json_loader.load()

    all_docs = pdf_docs + json_docs
    print(f"Loaded {len(pdf_docs)} PDF pages and {len(json_docs)} JSON threat intel entries.")
    return all_docs

# 3. Chunking with "HD" Overlap
raw_knowledge = load_all_cyber_knowledge(DATA_DIR)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(raw_knowledge)

print(f"Final Knowledge Base Size: {len(chunks)} chunks.")

STIX JSON saved to: /content/drive/MyDrive/CyberRAG_Project/data/threat_intel_play_ransomware_2025.json
Scanning Knowledge Base
Loaded 516 PDF pages and 5 JSON threat intel entries.
Final Knowledge Base Size: 1687 chunks.


# 3. The Vector Database (Persistence & GPU)

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Building Vector Database")

# BGE-Large v1.5 (Top of the MTEB Leaderboard)
# I use Colab Pro GPU (cuda) to make this 10x faster
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# Initialize Chroma and save it to your Drive (DB_DIR)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=DB_DIR
)

print(f"Vector Database persisted to: {DB_DIR}")

Building Vector Database


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Database persisted to: /content/drive/MyDrive/CyberRAG_Project/vector_store


# 4. Designing the Hybrid Retriever

In [11]:
!pip install -q -U langchain-classic

In [12]:
# Updated import for 2026 LangChain structure
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

print("Initializing Hybrid Retrieval System")

# 1. The Keyword Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

# 2. The Semantic Retriever (Vector)
# Ensure vector_db was successfully created in Step 4
vector_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 3. The Ensemble (Hybrid)
# This combines results using Reciprocal Rank Fusion (RRF)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

print("Hybrid Retriever is now online!")

Initializing Hybrid Retrieval System
Hybrid Retriever is now online!


# 5. The "Internal Sanity Check" (Retrieval Quality)

In [13]:
test_query = "What are the specific ISO 27001 governance requirements for ransomware incident response?"

# Fetch results
retrieved_docs = hybrid_retriever.invoke(test_query)

print(f"Query: {test_query}\n")
for i, doc in enumerate(retrieved_docs):
    source = doc.metadata.get('source', 'Unknown')
    print(f"Result {i+1} | Source: {os.path.basename(source)}")
    print(f"Content Snippet: {doc.page_content[:400]}...\n")

Query: What are the specific ISO 27001 governance requirements for ransomware incident response?

Result 1 | Source: cisa_ransomware_best_practices.pdf
Content Snippet: • Create, maintain, and regularly exercise a basic cyber incident response plan (IRP) and 
associated communications plan that includes response and notification procedures for 
ransomware and data extortion/breach incidents [CPG 2.S]. Ensure a hard copy of the plan and 
an offline version is available. 
o Provide data breach notifications to third parties and regulators consistent with law. 
o En...

Result 2 | Source: Copy of nist_incident_handling_historical.pdf
Content Snippet: determine how serious the attack is. The incident will be prioritized, and the incident handlers 
will take action to ensure that the progress of the incident is halted and that the affected systems 
return to normal operation as soon as possible. 
3. What is incident response? 
The terms “incident handling” and “incident response” are synony

# 6. Generation Module (Connecting the LLM)

## 6.1 Loading the LLM

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# Using Mistral-7B-v0.3: No gating/approval required, excellent for RAG
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading Mistral-7B (Open Access)")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load Model with 4-bit quantization for efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True
)

# Create the LangChain-compatible pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1, # Keep low for "Faithfulness"
    top_p=0.95,
    repetition_penalty=1.15,
    return_full_text=False # Ensures we only get the answer, not the prompt back
)

llm = HuggingFacePipeline(pipeline=pipe)
print("LLM loaded successfully without gating issues!")

Loading Mistral-7B (Open Access)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

TypeError: MistralForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

In [17]:
import json
import os

# This saves the current notebook state to a file in /content
# Replace the name below with your desired filename
filename = "assignment3_a1815352_sellaiya.ipynb"

# We use a shell command to 'save' the current notebook from the Colab internal path
!cp "/notebooks/{filename}" "/content/{filename}" 2>/dev/null || echo "Manual save needed: Go to File > Download > .ipynb and then upload it to the sidebar."

Manual save needed: Go to File > Download > .ipynb and then upload it to the sidebar.
